# Fine-tuning YAMNet on ESC-50

Transfer learning pipeline that adapts Google's YAMNet audio classifier to the
[ESC-50](https://github.com/karolpiczak/ESC-50) dataset (2,000 clips, 50 environmental sound classes),
then exports a TensorFlow Lite model for on-device inference in an Android app.

**Approach:** YAMNet is used as a frozen feature extractor. Each audio clip is converted into
1024-dimensional embeddings, and a small dense classifier is trained on top. This is far faster
than training from scratch and works well with a few thousand samples.

**Runtime:** Set to GPU via *Runtime → Change runtime type → T4 GPU* (optional but faster).

## 1. Setup

In [ ]:
!pip install -q tensorflow tensorflow_hub tensorflow_io

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

print('TensorFlow', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 2. Download ESC-50

2,000 five-second clips, 40 per class, sampled at 44.1 kHz.

In [ ]:
!wget -q https://github.com/karoldvl/ESC-50/archive/master.zip -O esc50.zip
!unzip -q esc50.zip
!rm esc50.zip

ESC50_DIR = 'ESC-50-master'
AUDIO_DIR = os.path.join(ESC50_DIR, 'audio')
meta = pd.read_csv(os.path.join(ESC50_DIR, 'meta', 'esc50.csv'))

print(f'{len(meta)} clips, {meta.category.nunique()} classes')
meta.head()

## 3. Load YAMNet

YAMNet expects **16 kHz mono float32** audio in the range −1…1. ESC-50 is 44.1 kHz,
so every clip must be resampled — the same conversion the Android app performs at runtime.

In [ ]:
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
TARGET_SR = 16000

# Sanity check: YAMNet returns (scores, embeddings, spectrogram)
test_wav = np.zeros(TARGET_SR, dtype=np.float32)
scores, embeddings, spectrogram = yamnet_model(test_wav)
print('scores    ', scores.shape)      # (frames, 521)
print('embeddings', embeddings.shape)  # (frames, 1024)

## 4. Data preparation

Decode each WAV, convert to mono, resample to 16 kHz, and normalize.

In [ ]:
import tensorflow_io as tfio

def load_wav_16k_mono(path):
    """Read a WAV file and return a 16 kHz mono float32 tensor."""
    contents = tf.io.read_file(path)
    wav, sample_rate = tf.audio.decode_wav(contents, desired_channels=1)
    wav = tf.squeeze(wav, axis=-1)
    sample_rate = tf.cast(sample_rate, tf.int64)
    wav = tfio.audio.resample(wav, rate_in=sample_rate, rate_out=TARGET_SR)
    return wav

# Build label mapping
classes = sorted(meta.category.unique())
class_to_id = {c: i for i, c in enumerate(classes)}
meta['label'] = meta.category.map(class_to_id)

print(f'{len(classes)} classes')
print(classes[:10], '...')

### Train / validation / test split

ESC-50 ships with five predefined folds. Using them (rather than a random split) prevents
clips from the same source recording appearing in both training and test sets.

In [ ]:
meta['path'] = meta.filename.apply(lambda f: os.path.join(AUDIO_DIR, f))

train_df = meta[meta.fold.isin([1, 2, 3])]
val_df   = meta[meta.fold == 4]
test_df  = meta[meta.fold == 5]

print(f'train {len(train_df)}  val {len(val_df)}  test {len(test_df)}')

## 5. Extract YAMNet embeddings

YAMNet slices audio into ~0.96 s frames and emits one 1024-d embedding per frame.
Each 5-second clip therefore produces several embeddings, all sharing the clip's label —
which conveniently multiplies the effective size of the training set.

In [ ]:
def extract_embeddings(df, desc=''):
    X, y = [], []
    total = len(df)
    for i, (path, label) in enumerate(zip(df.path.values, df.label.values)):
        wav = load_wav_16k_mono(path)
        _, embeddings, _ = yamnet_model(wav)
        emb = embeddings.numpy()
        X.append(emb)
        y.append(np.full(emb.shape[0], label))
        if (i + 1) % 200 == 0:
            print(f'  {desc} {i + 1}/{total}')
    return np.concatenate(X), np.concatenate(y)

print('Extracting embeddings (a few minutes)...')
X_train, y_train = extract_embeddings(train_df, 'train')
X_val,   y_val   = extract_embeddings(val_df,   'val')
X_test,  y_test  = extract_embeddings(test_df,  'test')

print('train', X_train.shape, 'val', X_val.shape, 'test', X_test.shape)

## 6. Train the classifier head

A small dense network over the frozen embeddings. Dropout guards against overfitting,
which matters because frames from the same clip are highly correlated.

In [ ]:
NUM_CLASSES = len(classes)

classifier = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,), dtype=tf.float32, name='embedding'),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
], name='esc50_classifier')

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
classifier.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True,
                                     monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor='val_loss'),
]

history = classifier.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=64,
    callbacks=callbacks,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.set_xlabel('epoch'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history.history['loss'], label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss'); ax2.set_xlabel('epoch'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Evaluate

Frame-level accuracy first, then clip-level accuracy — averaging predictions across a clip's
frames is what an application would actually do, and it scores noticeably higher.

In [ ]:
frame_loss, frame_acc = classifier.evaluate(X_test, y_test, verbose=0)
print(f'Frame-level test accuracy: {frame_acc:.3f}')

# Clip-level: average the softmax outputs over each clip's frames
clip_correct = 0
for path, label in zip(test_df.path.values, test_df.label.values):
    wav = load_wav_16k_mono(path)
    _, emb, _ = yamnet_model(wav)
    probs = classifier.predict(emb.numpy(), verbose=0).mean(axis=0)
    if np.argmax(probs) == label:
        clip_correct += 1

print(f'Clip-level test accuracy:  {clip_correct / len(test_df):.3f}')

In [ ]:
y_pred = classifier.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=classes, zero_division=0))

## 8. Combine YAMNet + classifier into one model

The Android app should receive raw waveform samples and get class scores back, without
having to run two models itself. Wrapping both into a single graph keeps the app simple.

In [ ]:
class CombinedModel(tf.Module):
    def __init__(self, yamnet, classifier):
        super().__init__()
        self.yamnet = yamnet
        self.classifier = classifier

    @tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)])
    def __call__(self, waveform):
        _, embeddings, _ = self.yamnet(waveform)
        probs = self.classifier(embeddings)
        return {'scores': tf.reduce_mean(probs, axis=0)}

combined = CombinedModel(yamnet_model, classifier)
tf.saved_model.save(combined, 'combined_saved_model')
print('Saved.')

## 9. Convert to TensorFlow Lite

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model('combined_saved_model')
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,   # YAMNet uses some ops without TFLite equivalents
]
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('esc50_yamnet.tflite', 'wb') as f:
    f.write(tflite_model)

with open('esc50_labels.txt', 'w') as f:
    f.write('\n'.join(classes))

size_mb = len(tflite_model) / 1e6
print(f'esc50_yamnet.tflite  {size_mb:.1f} MB')
print(f'esc50_labels.txt     {len(classes)} labels')

## 10. Verify the TFLite model

Always test the converted model before shipping it — quantization and op selection can
change behaviour relative to the Keras version.

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_model)
runner = interpreter.get_signature_runner()

correct = 0
sample = test_df.sample(50, random_state=0)
for path, label in zip(sample.path.values, sample.label.values):
    wav = load_wav_16k_mono(path).numpy()
    out = runner(waveform=wav)
    pred = np.argmax(out['scores'])
    correct += int(pred == label)

print(f'TFLite accuracy on 50 test clips: {correct / 50:.2f}')

## 11. Download

Place `esc50_yamnet.tflite` in the Android project at
`android/app/src/main/assets/` and update `MODEL_FILE` in `AudioClassifierModule.java`.

In [ ]:
from google.colab import files
files.download('esc50_yamnet.tflite')
files.download('esc50_labels.txt')

---

## Notes

- **Why freeze YAMNet.** With 2,000 clips, fine-tuning 3.7M parameters would overfit badly.
  Training only the head keeps the learned audio representation intact and converges in minutes.
- **Frames vs clips.** YAMNet emits one embedding per ~0.96 s frame, so each 5-second clip
  yields several training examples. At inference the frame predictions are averaged, which
  smooths out momentary confusion and raises accuracy.
- **Fold-based splitting.** ESC-50 clips are cut from longer source recordings; a random split
  would leak near-duplicate audio between train and test and inflate the score.
- **SELECT_TF_OPS.** YAMNet's preprocessing uses ops that have no TFLite builtin equivalent,
  so the Flex delegate is required. This increases model size and means the Android build
  needs the `tensorflow-lite-select-tf-ops` dependency.
- **Possible improvements.** Data augmentation (time shift, noise, pitch), per-class threshold
  tuning, and full fine-tuning of YAMNet's upper layers with a very low learning rate.